# 8 · Collect Linear Probing Results
Reads the `results-linear.csv` files produced by notebook 7 and assembles summary tables.

- **`df_ue_train`** — train and eval from the pre-training split.
- **`df_ue_train_on_eval`** — train from pre-training split, eval from held-out split.
- **`accuracy_dict`** — per-experiment accuracy array from a pre-computed `.npy` file.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
CKP_PATH = './metadata/DINOv2_checkpoint/'

EVAL_DIRS_BASE = ('EMD_Resolution_ue_train', 'MolecularWeight_kDa_ue_train', 'ImagePixelSize_ue_train',
                  'DefocusV_ue_train', 'MaxDiameter_ue_train', 'Symmetry_ue_train')

EVAL_DIRS_TRAIN = [i for i in EVAL_DIRS_BASE]
EVAL_DIRS_TRAIN_RegressionOutProjector = [f'{i}_RegressionOutProjector' for i in EVAL_DIRS_BASE]

EVAL_DIRS_TRAIN_ON_EVAL = [f'{i}_on_eval' for i in EVAL_DIRS_BASE]
EVAL_DIRS_TRAIN_ON_EVAL_RegressionOutProjector = [f'{i}_on_eval_RegressionOutProjector' for i in EVAL_DIRS_BASE]

In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Helper

In [3]:
def load_accuracy_values(file_path):
    accuracies = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("{"):
                data = json.loads(line)
                accuracies.append(data["best_classifier"]["accuracy"])

    return accuracies

def load_results(ckp_path, eval_dirs):
    """
    Read results-linear.csv for each task directory.
    Missing or unparseable files produce NaN so they are visible as gaps
    and are excluded from the mean automatically.
    """
    df = pd.DataFrame(index=sorted(eval_dirs), columns=['value'], dtype=float)
    for d in eval_dirs:
        json_path = os.path.join(ckp_path, f'Linear_{d}', 'results_eval_linear.json')
        try:
            df.loc[d, 'value'] = np.round(100*np.max(load_accuracy_values(json_path)),1)
        except FileNotFoundError:
            print(f'  [missing]     {json_path}')
        except (ValueError, IndexError) as exc:
            print(f'  [parse error] {json_path}: {exc}')
    df.loc['mean'] = np.round(df['value'].mean(),1)
    return df

## 1 · `(i) train` results

In [4]:
df_train = load_results(CKP_PATH, EVAL_DIRS_TRAIN)
df_train

,value
DefocusV_ue_train,77.6
EMD_Resolution_ue_train,86.1
ImagePixelSize_ue_train,93.2
MaxDiameter_ue_train,93.8
MolecularWeight_kDa_ue_train,88.9
Symmetry_ue_train,93.6
mean,88.9


## 2 · `(ii) train_on_eval` results

In [5]:
df_train_on_eval = load_results(CKP_PATH, EVAL_DIRS_TRAIN_ON_EVAL)
df_train_on_eval

,value
DefocusV_ue_train_on_eval,68.2
EMD_Resolution_ue_train_on_eval,65.3
ImagePixelSize_ue_train_on_eval,67.7
MaxDiameter_ue_train_on_eval,76.9
MolecularWeight_kDa_ue_train_on_eval,57.5
Symmetry_ue_train_on_eval,60.9
mean,66.1


## 3 · `(iii) train_regressed` results

In [6]:
df_train_regressed = load_results(CKP_PATH, EVAL_DIRS_TRAIN_RegressionOutProjector)
df_train_regressed

,value
DefocusV_ue_train_RegressionOutProjector,79.8
EMD_Resolution_ue_train_RegressionOutProjector,88.3
ImagePixelSize_ue_train_RegressionOutProjector,96.4
MaxDiameter_ue_train_RegressionOutProjector,94.3
MolecularWeight_kDa_ue_train_RegressionOutProjector,91.1
Symmetry_ue_train_RegressionOutProjector,94.3
mean,90.7


## 3 · `(iv) train_on_eval_regressed` results

In [7]:
df_train_on_eval_regressed = load_results(CKP_PATH, EVAL_DIRS_TRAIN_ON_EVAL_RegressionOutProjector)
df_train_on_eval_regressed

,value
DefocusV_ue_train_on_eval_RegressionOutProjector,73.1
EMD_Resolution_ue_train_on_eval_RegressionOutProjector,67.9
ImagePixelSize_ue_train_on_eval_RegressionOutProjector,71.5
MaxDiameter_ue_train_on_eval_RegressionOutProjector,79.9
MolecularWeight_kDa_ue_train_on_eval_RegressionOutProjector,59.6
Symmetry_ue_train_on_eval_RegressionOutProjector,61.9
mean,69.0
